# Track 01 — EEG-to-Image · pipeline proof

**What this notebook proves:** that we can pull the data, pull a model, train it, evaluate it
with the competition's metric, and package a submission — end to end, on Colab, without surprises.

**What it does not do:** chase performance. Every model here is a stock baseline.

---

### The task

Given one EEG epoch recorded while a participant viewed a natural image, predict a **1536-d
embedding** in frozen `facebook/dinov2-giant` space, then rank held-out candidate images by
similarity. Training and test images do not overlap — the shift is *cross-stimulus*.

| | |
|---|---|
| Ranking metric | Top-5 retrieval accuracy against the full held-out gallery |
| NeuralBench key | `test/full_retrieval/top5_acc_subject-agg` |
| Target space | `facebook/dinov2-giant`, relative depth 0.6667, mean token pooling, imsize 518 |
| Hidden cohort | 11 participants, 32 ch @ 256 Hz (Alljoined) |

⚠️ `val/batch_top5_acc` ranks **within a batch** only. It reads far higher and is not comparable
to the competition score. Never quote it.

On the hardware: the challenge site describes the cohort as 32-channel Emotiv. The Alljoined-1.6M
paper (arXiv:2508.18571) describes a 32-channel consumer-grade **wet** electrode system at ~$2.2k,
and the recordings carry standard 10-10 electrode names rather than vendor labels. Same count and
rate either way; the naming matters because it means a standard montage resolves cleanly.

## Two things to know before running

**1. Subsetting applies to Stage 0 only.** `FRACTION` and the byte budget govern how many *records*
EEGDash pulls. Stage 1 uses NeuralBench, whose downloader works at corpus granularity with no subset
option, so it fetches all of Alljoined-1.6M regardless. Inside Stage 1 the only data reduction is
`-d/--debug`, which NeuralBench applies itself.

**2. Track 1 subsetting has a trap.** The score is retrieval against the held-out gallery, so chance
is `k / gallery_size`. Shrinking the gallery inflates the score for reasons unrelated to the model:

| gallery | chance Top-5 |
|---|---|
| 16,740 | 0.03% |
| 200 | 2.50% |
| 10 | **50.00%** |

So Stage 0 subsets the training side and leaves the test gallery whole, and `assert_gallery_intact`
fails the run if a subset reaches the test split. Every Top-5 number is reported with its gallery
size attached, because one without the other means nothing.

> Everything below has been confirmed against a real run except where a cell says otherwise.

---

# Setup — run this first, everything below depends on it

Built for **Run All**. The next cell prompts for a run mode and sets everything from your answer.
Press Enter at each prompt to take the default.

Run All pauses at that cell until you answer, which is the intent. If the prompt is left unanswered
long enough for the runtime to drop, just run again.

| flag | off (default) | on |
|---|---|---|
| `RUN_HEAVY` | Stage 0 only: ~60 MB, CPU, minutes | adds the 7.7 GB download, prepare, and debug run |
| `RUN_FULL_TRAIN` | no full training | adds the full EEGNet baseline, hours |

With both off, Run All completes the whole discovery path and skips every expensive cell with a
printed reason. Nothing errors, nothing blocks on input.

Drive mounts **before** the installs, deliberately: installing NeuralBench downgrades packages Colab
pins, including `requests`, and `google.colab` imports can fail afterwards.

In [ ]:
# Prompts for the run mode. Everything below reads the flags this sets.

MODES = {
    '0': ('discovery', 'CPU, ~60 MB, minutes. Data access, montage, events.'),
    '1': ('smoke',     'GPU, ~4.4 GB. Whole pipeline + submission path on tiny data.'),
    '2': ('download',  'GPU, ~13 GB, hours. Track 1 corpus, prepare cache, --debug.'),
    '3': ('train',     'GPU, many hours. Adds the full EEGNet baseline.'),
}


def ask(prompt, default):
    try:
        return input(f'{prompt} [{default}]: ').strip() or default
    except (EOFError, OSError):
        print(f'{prompt}: no stdin, using {default}')
        return default


print('Run mode')
print('-' * 60)
for key, (name, desc) in MODES.items():
    print(f'  {key}  {name:<10} {desc}')
print('-' * 60)

choice = ask('Mode', '0')
while choice not in MODES:
    print(f"  '{choice}' is not one of {', '.join(MODES)}.")
    choice = ask('Mode', '0')

USE_DRIVE = ask('Mount Google Drive? y/n', 'y').lower().startswith('y')

raw_budget = ask('Stage 0 download budget in MB', '60')
try:
    BUDGET_MB = int(float(raw_budget))
except ValueError:
    print(f"  '{raw_budget}' is not a number, using 60")
    BUDGET_MB = 60

FRACTION = 0.02   # Stage 0 group subsetting; NeuralBench ignores it
SEED = 33

RUN_SMOKE = int(choice) >= 1        # NeuralBench end-to-end on a 1.5 GB dataset
RUN_HEAVY = int(choice) >= 2        # Track 1 corpus: 7.7 GB + DINOv2
RUN_FULL_TRAIN = int(choice) >= 3   # full baseline

print()
print(f'mode           : {choice} - {MODES[choice][0]}')
print(f'RUN_SMOKE      : {RUN_SMOKE}')
print(f'RUN_HEAVY      : {RUN_HEAVY}')
print(f'RUN_FULL_TRAIN : {RUN_FULL_TRAIN}')
print(f'mount drive    : {USE_DRIVE}')
print(f'stage 0 budget : {BUDGET_MB} MB, fraction {FRACTION}, seed {SEED}')
if not RUN_SMOKE:
    print('\nEverything past discovery will skip with a printed reason. Nothing will error.')
elif not RUN_HEAVY:
    print('\nTrack 1 corpus cells will skip. The smoke test still proves the full loop.')

In [ ]:
import os, sys, json, subprocess, shutil
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/neurips26-eeg')
else:
    ROOT = Path('/content/neurips26-eeg')

DATA_DIR = ROOT / 'data'
SAVE_DIR = ROOT / 'results'
CACHE_LOCAL = Path('/content/nb_cache')     # rebuildable, so keep it off slow Drive
EEGDASH_CACHE = Path('/content/eegdash_cache')
for d in (DATA_DIR, SAVE_DIR, CACHE_LOCAL, EEGDASH_CACHE):
    d.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(ROOT / 'hf')
os.environ['EEGDASH_CACHE_DIR'] = str(EEGDASH_CACHE)
CACHE = EEGDASH_CACHE
print('ROOT      :', ROOT)
print('free (data):', round(shutil.disk_usage(DATA_DIR).free / 1e9, 1), 'GB')
print('free (local):', round(shutil.disk_usage('/content').free / 1e9, 1), 'GB')

In [ ]:
REPO = 'https://github.com/AGRamirezz/Neurips26-eeg-foundation-model.git'
SRC = Path('/content/repo/src')

if Path('../src/miniload.py').exists():
    SRC = Path('../src').resolve()
elif not SRC.exists():
    subprocess.run(['git', 'clone', '-q', REPO, '/content/repo'], check=True)
sys.path.insert(0, str(SRC))
print('helpers:', sorted(f.name for f in SRC.glob('*.py')))

In [ ]:
%pip install -q 'eegdash>=0.9.1' neuralbench==0.3.1 'transformers' 'torchvision' 'pillow'

In [ ]:
# Fail loudly here rather than 40 cells later.
import importlib
for mod in ('eegdash', 'neuralbench', 'mne'):
    try:
        importlib.import_module(mod)
        print(f'{mod:12} ok')
    except ImportError as e:
        raise SystemExit(f'{mod} did not install: {e}. Restart the runtime and re-run Setup.')

print('python  :', sys.version.split()[0])
assert sys.version_info >= (3, 12), 'NeuralBench needs Python >= 3.12'

### NeuralBench storage

Environment variables are **not** read. With no terminal to prompt from, NeuralBench silently
defaults `DATA_DIR`, `CACHE_DIR` and `SAVE_DIR` to `/tmp/neuralbench`, which Colab discards on
restart. The mechanism is a config file at `~/.neuralbench/config.json`.

`DATA_DIR` goes to Drive because a 7.7 GB download is expensive to refetch. `CACHE_DIR` stays on
local disk because Drive is slow for many small files and the cache rebuilds in minutes.

In [ ]:
import time
CFG_PATH = Path.home() / '.neuralbench' / 'config.json'
CFG_PATH.parent.mkdir(parents=True, exist_ok=True)
CFG_PATH.write_text(json.dumps({
    'USER': os.environ.get('USER', 'root'),
    'ENTITY_NAME': os.environ.get('USER', 'root'),
    'PROJECT_NAME': 'neuralbench',
    'DATA_DIR': str(DATA_DIR),
    'CACHE_DIR': str(CACHE_LOCAL),
    'SAVE_DIR': str(SAVE_DIR),
    'WANDB_HOST': '', 'SLURM_PARTITION': '', 'SLURM_CONSTRAINT': '',
    'N_CPUS': os.cpu_count() or 2, 'CLUSTER': 'auto',
}, indent=2))
os.environ['NEURALBENCH_CONFIG'] = str(CFG_PATH)

# The CLI runs as a subprocess, so check what a fresh interpreter resolves.
out = subprocess.run([sys.executable, '-c',
    'from neuralbench import config_manager as c; import json; print(json.dumps(c.get_config()))'],
    capture_output=True, text=True)
resolved = json.loads(out.stdout.strip().splitlines()[-1])
for k in ('DATA_DIR', 'CACHE_DIR', 'SAVE_DIR'):
    print(f'{k:10} {resolved[k]}')
assert '/tmp/' not in resolved['DATA_DIR'], 'config not picked up: DATA_DIR is still ephemeral'
print('\nconfig ok')

---

# Stage 0 — smoke test on a few recordings

**Run this first.** CPU only, no GPU, no Drive. It works on a laptop.

Alljoined-1.6M is on EEGDash as **`nm000134`**: 20 subjects, 1525 recordings, 32 ch @ 256 Hz. One
317 s recording is about 6 MB, so a 60 MB budget buys roughly six of them, spread across six
subjects. Same corpus and hardware as the hidden evaluation cohort.

(EEGDash reports the corpus at 8.8 GB and 129 h; the challenge dataset table says 7.7 GB and 130 h.
Same data, different accounting. Neither figure matters at this scale.)

Record selection is the only way to touch a large corpus cheaply, because NeuralBench's own
downloader has no subset option.

Stage 0's job is **discovery**: print what the data actually looks like, so the stubbed cells in
Stage 1 can be written against reality rather than guessed from docs.

⚠️ Alljoined-1.6M is **CC-BY-NC-ND-4.0**, non-commercial *and* no-derivatives. Stricter than the
other Track 1 corpora. Worth checking before anything derived from it is published.

### 0.1 — What is in the dataset

Query first, download nothing. This tells us the subject/session/task vocabulary to select on.

In [ ]:
from eegdash import EEGDash

DATASET = 'nm000134'   # Alljoined-1.6M

records = EEGDash().find({'dataset': DATASET})
print('records:', len(records))

# Keep the converted BIDS files, not the original source format.
records = [r for r in records if r['bids_relpath'].startswith('sub-')]
print('bids records:', len(records))
print('\nfields on a record:')
print(json.dumps({k: str(v)[:60] for k, v in records[0].items()}, indent=2))

In [ ]:
# Vocabulary we can select on.
from collections import Counter
for field in ('subject', 'session', 'task', 'run'):
    vals = Counter(r.get(field) for r in records)
    print(f'{field:>8}: {len(vals)} unique -> {sorted(str(v) for v in vals)[:8]}')

### 0.2 — Pull a few recordings, under a byte budget

`select_under_budget` reads `ntimes` and `nchans` from the record metadata, so the size of a
selection is known before anything transfers.

`spread_by='subject'` matters more than it looks. Record lists arrive grouped by subject, so taking
the first six in order gives six recordings from one person: enough to prove the code runs, useless
for seeing whether it handles variation. Interleaving costs identical bytes.

In [ ]:
from miniload import select_under_budget, estimate_mb, host_limits, epochs_mb, targets_mb

print('host:', {k: (v if isinstance(v, str) else round(v, 1)) for k, v in host_limits().items()})

# Budget first, download second. Spread across subjects so the batch exercises
# variation, not just one person's recordings.
sel = select_under_budget(records, budget_mb=BUDGET_MB, spread_by='subject')
picked = sel.records

print(f'\n{len(picked)} records, ~{sel.est_mb:.0f} MB est '
      f'({len({r["subject"] for r in picked})} subjects, {sel.skipped_over_budget} skipped)')
for r in picked:
    print(f"  {estimate_mb(r):5.1f} MB  {r['bids_relpath']}")

**Memory, measured rather than guessed.** One 317 s recording at 32 ch / 256 Hz is 10.4 MB as
float32. Epoched into 1 s windows it becomes ~42 MB per recording, and the full set of 16,740
DINOv2-giant targets is 103 MB (which matches the ~100 MB the NeuralBench docs quote for
THINGS-EEG2, a useful cross-check).

None of that is near a Colab limit. The whole corpus held in RAM would be ~15.6 GB, which is why we
do not hold it. The one item of consequence is the DINOv2-giant checkpoint at ~4.5 GB; on a GPU
runtime it lands in VRAM, on a CPU runtime it competes with everything else.

In [ ]:
from eegdash import EEGDashDataset

ds = EEGDashDataset(records=picked, cache_dir=CACHE)
print(ds.description.to_string(index=False))

### 0.3 — Inspect one recording

Signal shape, sampling rate, channel names, and the event schema.

**Outcome:** 32 channels at 256 Hz, 317 s, and `montage_present: False`. The events turned out not
to carry per-image identity at all, which 0.5 and 0.6 follow up.

In [ ]:
raw = ds.datasets[0].raw
print('sfreq   :', raw.info['sfreq'])
print('n_chans :', len(raw.ch_names))
print('channels:', raw.ch_names)
print('duration:', raw.n_times / raw.info['sfreq'], 's')
print('has montage positions:', raw.get_montage() is not None)

### 0.4 — Consolidated report

One cell summarising the run. This is what resolved the montage gap, the `02old` session, and the
event schema.

In [ ]:
import platform, sys

report = {
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'n_records_total': len(records),
    'select_fields': {f: sorted({str(r.get(f)) for r in records})[:10]
                      for f in ('subject', 'session', 'task', 'run')},
    'record_keys': list(records[0].keys()),
    'sfreq': raw.info['sfreq'],
    'ch_names': raw.ch_names,
    'n_annotations': len(raw.annotations),
    'annotation_sample': list(raw.annotations.description[:3]),
    'montage_present': raw.get_montage() is not None,
}
print(json.dumps(report, indent=2, default=str))

---

## 0.5 — Fixes from the first run

Three findings from the Stage 0 report, in order of importance.

**Montage is absent.** `montage_present: False`. REVE encodes 3-D electrode coordinates, so with no
montage every channel becomes `INVALID_VALUE` and the model's main advantage silently disappears.
The 32 names are standard 10-10 and all resolve, verified offline at 32/32 with no NaNs. Use
**`standard_1020`**: `standard_1005` resolves equally well but is deprecated from MNE 1.14. Same
pattern the stock Track 3 config uses for Sleep-EDF.

**Session `02old` exists** alongside `01`-`04`. A superseded duplicate. Exclude it, along with
records flagged `_has_missing_files`.

**Events are not in `raw.annotations`.** 61 annotations over 317 s, a median gap of 5.6 s, far too
sparse for a corpus averaging ~1050 image trials per recording.

In [ ]:
# Record filtering, and a size budget computed before anything downloads.
EXCLUDE_SESSIONS = {'02old'}

clean = [
    r for r in records
    if not r.get('_has_missing_files')
    and r.get('session') not in EXCLUDE_SESSIONS
]
print(f'{len(records)} records -> {len(clean)} after filtering')

def est_mb(r):
    # float32 per sample per channel; a rough but useful pre-download budget.
    n = r.get('ntimes') or 0
    c = r.get('nchans') or 0
    return n * c * 4 / 1e6

tot = sum(est_mb(r) for r in clean)
print(f'full filtered set: ~{tot/1000:.1f} GB across {len(clean)} records')
print(f'median record: ~{sorted(est_mb(r) for r in clean)[len(clean)//2]:.1f} MB')

In [ ]:
# Set the montage explicitly and confirm every channel resolves to real coordinates.
import numpy as np, mne

MONTAGE = 'standard_1020'   # 32/32 resolve; standard_1005 is deprecated from MNE 1.14

raw.set_montage(mne.channels.make_standard_montage(MONTAGE), match_case=False)

pos = raw.get_montage().get_positions()['ch_pos']
P = np.array([pos[c] for c in raw.ch_names])
assert not np.isnan(P).any(), 'some channels still have no coordinates'
print(f'{len(P)}/{len(raw.ch_names)} channels positioned, no NaNs')
print(f'posterior (y<0): {(P[:,1]<0).sum()}  anterior (y>0): {(P[:,1]>0).sum()}')
# Heavily posterior - a visual-task montage, unlike the frontal/temporal-heavy
# clinical layouts most EEG foundation models pretrain on.

### Annotations decoded

`description` holds **four** comma-separated fields: `<condition>,<stim_id>,<block>,<trial>`. The
TSV writes it as `behav,3,-1,21`, which reads as a three-field record with a decimal comma until you
count. Conditions are `behav` (ids 2, 3) and `oddball` (id 16740); block is `-1` throughout.

Read that way the trial index is monotonic: 21, 42, 63, 84, then 381, 402, 423, mostly stepping by
21. A behavioural probe lands every 21 image trials, so 61 markers spans roughly 1280 trials. The
image presentations happened; this file does not enumerate them.

`16740` is the THINGS catalogue size, so the oddball id is a sentinel rather than a real image
index. The corpus is part of the THINGS initiative (arXiv:2508.18571), recorded on a 32-channel
consumer-grade **wet** system, which is why the channel names are standard 10-10 rather than a
vendor's own labels.

⚠️ Use `raw.annotations.onset` for seconds. `to_data_frame()` returns absolute datetimes here
because `meas_date` is set, which is useless for epoching.

In [ ]:
import pandas as pd

ann = pd.DataFrame({
    'onset_s': raw.annotations.onset,          # seconds, not datetimes
    'duration': raw.annotations.duration,
    'description': raw.annotations.description,
})
# Four fields: condition, stimulus id, block, trial index. The TSV writes the last
# two with a comma between them, which reads as a decimal comma until you count.
parts = ann['description'].str.split(',', expand=True)
ann[['condition', 'stim_id', 'block', 'trial']] = parts.iloc[:, :4]
for c in ('stim_id', 'block', 'trial'):
    ann[c] = pd.to_numeric(ann[c], errors='coerce')

print(ann.head(12).to_string())
print('\nconditions:', ann['condition'].value_counts().to_dict())
print('blocks:', ann['block'].unique())
print('trial index: %s -> %s, monotonic=%s'
      % (ann['trial'].min(), ann['trial'].max(), ann['trial'].is_monotonic_increasing))

### Find the stimulus identity

**Outcome: not present.** The `events.tsv` for `sub-01/ses-02/run-13` has the same 61 rows as the
annotations, columns `onset, duration, trial_type, value, sample`, and no per-image entries. Cells
below are kept because they are the right probe to re-run against other recordings.

In [ ]:
import pandas as pd

ev_files = sorted(CACHE.rglob('*_events.tsv'))
print(f'{len(ev_files)} events.tsv found')
for f in ev_files[:3]:
    print('  ', f.relative_to(CACHE))

assert ev_files, 'no events.tsv in cache - check what EEGDash actually downloaded'
ev = pd.read_csv(ev_files[0], sep='\t')
print(f'\nshape: {ev.shape}')
print('columns:', list(ev.columns))
print(ev.head(10).to_string())

In [ ]:
# Which column identifies the image? Look for one with many distinct values.
for c in ev.columns:
    n = ev[c].nunique()
    print(f'{c:>24}  {n:>6} unique  e.g. {list(ev[c].dropna().unique()[:4])}')

# Also: what are the 61 annotations, if not stimuli?
print('\nannotation descriptions:', raw.annotations.description[:20].tolist())

In [ ]:
# Any stimulus->file mapping at the dataset root (THINGS-EEG2 used stimuli.tsv).
for pat in ('stimuli.tsv', 'participants.tsv', '*_events.json', 'dataset_description.json'):
    for f in sorted(CACHE.rglob(pat))[:2]:
        print('--', f.relative_to(CACHE))
        if f.suffix == '.tsv':
            print(pd.read_csv(f, sep='\t').head(5).to_string())
        else:
            print(json.dumps(json.load(open(f)), indent=2)[:600])

---

## 0.6 — Where is the image stream? (closed)

Durations are unimodal: 255-409 s, median 308, one peak across all 1520 filtered records. There is
no separate class of image runs, so per-image identity is not a matter of picking different
recordings.

`events.tsv` for `sub-01/ses-02/run-13` holds 61 rows, all `behav` or `oddball`, matching the
annotations exactly. The trailing field is a cumulative trial counter stepping by 21, so ~1280 image
trials occurred without being enumerated.

**Closed rather than solved.** NeuralBench ships a registered `xu2025alljoined` config and does its
own target extraction, so Track 1 does not depend on us parsing this. Recorded as an open question
in `PLAN.md` §4b.

---

# Smoke test — everything, on tiny data, all local

One mode, two parts, run back to back. Clears bugs before any big download or any submission.

| part | proves | cost |
|---|---|---|
| **A** the pipeline | data load, target extraction, model load, train, eval | ~4.4 GB + DINOv2 |
| **B** the submission path | weight export, packaging, `predict` contract, scoring, timing | zero |

Two parts only because they are two tools. Everything in B is benchopt and `CompetSolver`, which no
`neuralbench` command touches on any task.

Runs at mode **1** and above.

## Part A — the pipeline

`eeg image --dataset xu2024alljoined`: Alljoined-1, the smallest registered dataset for Track 1's
own task. ~4.4 GB, plus ~4.5 GB for DINOv2-giant on first use.

Track 1's real task, so this exercises target extraction and retrieval scoring rather than standing
in with a classification task. It runs at 64 ch / 512 Hz against Track 1's 32 ch / 256 Hz, which
also puts montage handling through a channel-count change.

Covers: config, download, cache, target extraction, model build, train, eval. `chance` runs
alongside `eegnet` so a floor is visible and the metric is provably attached to something.

In [ ]:
if RUN_SMOKE:
    !neuralbench eeg image --dataset xu2024alljoined --download
else:
    print('skipped: choose mode 1 or higher')

In [ ]:
if RUN_SMOKE:
    t0 = time.time()
    !neuralbench eeg image --dataset xu2024alljoined --prepare
    print(f'prepare took {(time.time() - t0) / 60:.1f} min')
else:
    print('skipped: choose mode 1 or higher')

In [ ]:
if RUN_SMOKE:
    !neuralbench eeg image --dataset xu2024alljoined -m eegnet --debug
else:
    print('skipped: choose mode 1 or higher')

In [ ]:
if RUN_SMOKE:
    t0 = time.time()
    !neuralbench eeg image --dataset xu2024alljoined -m eegnet
    !neuralbench eeg image --dataset xu2024alljoined -m chance
    print(f'smoke train+chance took {(time.time() - t0) / 60:.1f} min')
else:
    print('skipped: choose mode 1 or higher')

**What a pass means.** Config resolved, the study downloaded and cached, a model built and
trained, and a metric came back above chance. Everything the Track 1 run needs, minus Track 1's
data. A failure here is a setup problem; a failure after this, with this green, is a data problem.

## Part B — is the submission path ready

Uses the **real EEG** pulled in Stage 0: real channel count, real sampling rate, real value scales.
Those are what actually break a `predict` contract, so inventing them would weaken the test.
Synthetic data is the fallback if Stage 0's download did not complete.

The one thing that cannot be real is the epoch-to-image correspondence, since Alljoined's per-image
identity is unresolved. That is useful rather than a compromise: with labels carrying no signal, the
correct result is retrieval **at chance**. Asserting that is stronger than asserting above-chance,
because a score above chance on random labels means a leak or a metric bug.

Part A already proves a model can learn. Part B proves the contract holds. No overlap.

In [ ]:
if RUN_SMOKE:
    import numpy as np, torch
    from harness import make_meta, run_local, chance_top_k, simulated

    WIN_S = 1.0
    if 'raw' in globals():
        sig = raw.get_data()                      # (n_chans, n_times), volts
        n_t = int(WIN_S * raw.info['sfreq'])
        n_w = sig.shape[1] // n_t
        X_real = (sig[:, :n_w * n_t]
                  .reshape(sig.shape[0], n_w, n_t)
                  .transpose(1, 0, 2).astype(np.float32))
        # Volts are ~1e-5, which silently underflows a naive model. Real preprocessing.
        mu = X_real.mean(axis=(0, 2), keepdims=True)
        sd = X_real.std(axis=(0, 2), keepdims=True) + 1e-12
        X_real = (X_real - mu) / sd
        fnames = list(getattr(raw, 'filenames', []) or [])
        source = f'real EEG from {Path(fnames[0]).name if fnames else "Stage 0"}'
    else:
        print('Stage 0 data unavailable, falling back to synthetic')
        X_real, _, _ = simulated(200, 32, 256, 384, seed=SEED)
        source = 'synthetic fallback'

    N, N_CH, N_T = X_real.shape
    print(f'{source}: X {X_real.shape}  '
          f'mean {X_real.mean():+.3f}  std {X_real.std():.3f}')
else:
    print('skipped: choose mode 1 or higher')

### Targets

Runs the real extraction path: load a vision model, batch, mean-pool, cache. `dinov2-small` at 88 MB
stands in for the competition's `dinov2-giant` at 4.5 GB. Same code path, 384-d instead of 1536-d.

Each window is assigned a gallery image by a seeded permutation. The assignment is arbitrary and
recorded as such; it exists so the retrieval task is well formed, not so it means anything.

Train and test galleries are disjoint, mirroring the cross-stimulus shift.

In [ ]:
if RUN_SMOKE:
    from transformers import AutoImageProcessor, AutoModel
    from PIL import Image

    DIM = 384
    n_gal = min(N, 200)          # 200 mirrors THINGS-EEG2's test gallery; chance = 5/200
    rng = np.random.default_rng(SEED)
    images = [Image.fromarray(rng.integers(0, 255, (64, 64, 3), dtype=np.uint8))
              for _ in range(n_gal)]

    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    proc = AutoImageProcessor.from_pretrained('facebook/dinov2-small')
    vis = AutoModel.from_pretrained('facebook/dinov2-small').to(dev).eval()

    t0 = time.time()
    embs = []
    with torch.no_grad():
        for s in range(0, n_gal, 32):
            batch = proc(images=images[s:s + 32], return_tensors='pt').to(dev)
            embs.append(vis(**batch).last_hidden_state.mean(dim=1).cpu().numpy())
    gallery = np.concatenate(embs).astype(np.float32)
    assert gallery.shape == (n_gal, DIM), gallery.shape
    print(f'embedded {n_gal} images -> {gallery.shape} in {time.time() - t0:.1f}s')

    # Arbitrary, seeded, recorded. Carries no signal by construction.
    assign = rng.permutation(N) % n_gal
    n_tr = int(0.7 * n_gal)
    tr_imgs, te_imgs = set(range(n_tr)), set(range(n_tr, n_gal))   # disjoint galleries

    tr = np.array([i for i in range(N) if assign[i] in tr_imgs])
    te = np.array([i for i in range(N) if assign[i] in te_imgs])
    X_tr, Y_tr = X_real[tr], gallery[assign[tr]]
    X_te = X_real[te]
    Y_te = gallery[n_tr:]                       # the held-out gallery
    truth_te = assign[te] - n_tr
    print(f'train {X_tr.shape} | test {X_te.shape} | held-out gallery {Y_te.shape}')
    print(f'chance top5 on this gallery: {chance_top_k(len(Y_te)):.2f}%')
else:
    print('skipped: choose mode 1 or higher')

### Train, then export

A deliberately small model, trained briefly. Under test is that the loop runs and the weights
round-trip to disk in a form `load_model` can read. It is not expected to learn anything, because
the labels carry no signal.

In [ ]:
if RUN_SMOKE:
    torch.manual_seed(SEED)

    model = torch.nn.Sequential(
        torch.nn.Flatten(),
        torch.nn.Linear(N_CH * N_T, 256),
        torch.nn.GELU(),
        torch.nn.Linear(256, DIM),
    ).to(dev)

    xb = torch.tensor(X_tr, device=dev)
    yb = torch.nn.functional.normalize(torch.tensor(Y_tr, device=dev), dim=1)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

    for epoch in range(10):
        opt.zero_grad()
        loss = 1 - torch.nn.functional.cosine_similarity(
            torch.nn.functional.normalize(model(xb), dim=1), yb).mean()
        loss.backward()
        opt.step()
        if epoch % 5 == 0 or epoch == 9:
            print(f'  epoch {epoch:>2}  cosine loss {loss.item():.4f}')

    SUB = Path('/content/submission_smoke')
    SUB.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), SUB / 'weights.pt')
    print(f'\nexported {(SUB / "weights.pt").stat().st_size / 1e6:.1f} MB')
else:
    print('skipped: choose mode 1 or higher')

### Package, load, predict, score, time

`submission.py` written against the documented contract. The only line that changes when the
starting kit ships is the import: `from harness import CompetSolver` becomes
`from compet_core.base_solver import CompetSolver`. The solver body stays as is.

In [ ]:
if RUN_SMOKE:
    (SUB / 'submission.py').write_text('''import torch

from harness import CompetSolver   # -> compet_core.base_solver once the kit ships


class _Wrapped(torch.nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net

    @torch.no_grad()
    def predict(self, X):
        return self.net(X)


class Solver(CompetSolver):
    name = "PipelineProof-T1"

    def load_model(self, meta):
        net = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(meta["n_chans"] * meta["n_times"], 256),
            torch.nn.GELU(),
            torch.nn.Linear(256, meta["n_outputs"]),
        )
        net.load_state_dict(torch.load(meta["weights_dir"] / "weights.pt",
                                       map_location=meta["device"]))
        return _Wrapped(net).to(meta["device"]).eval()
''')
    print((SUB / 'submission.py').read_text())
else:
    print('skipped: choose mode 1 or higher')

In [ ]:
if RUN_SMOKE:
    meta = make_meta(N_CH, N_T, DIM, float(raw.info['sfreq']) if 'raw' in globals() else 256.0,
                     SUB, device=dev)
    report = run_local(SUB, X_te, Y_te, truth_te, meta,
                       to_batch=lambda a: torch.tensor(a, device=dev))
    print(report)

    # Labels carry no signal, so the metric must land at chance. Above chance means a
    # leak or a broken metric; far below means the scoring path is wrong.
    p = report.chance_top5
    sigma = 100 * ((p / 100) * (1 - p / 100) / len(X_te)) ** 0.5
    dev_sigma = abs(report.top5 - p) / max(sigma, 1e-9)
    print(f'\ntop5 {report.top5:.2f}% vs chance {p:.2f}%  ({dev_sigma:.1f} sigma)')
    assert dev_sigma < 4, f'{dev_sigma:.1f} sigma from chance on random labels: leak or metric bug'
    assert report.budget_fraction() < 1.0, 'inference exceeds the 60 min budget'
    print('contract, metric and budget all pass')
else:
    print('skipped: choose mode 1 or higher')

### Block 13 — reproducibility record

Rule 4 wants every pretraining corpus declared with a compute estimate; Rule 7 re-runs the top
three from their committed config within +/-2 sigma. Adopting REVE means inheriting 92 datasets to
disclose. Cheaper to accumulate as you go than to reconstruct in November.

In [ ]:
if RUN_SMOKE:
    import datetime

    record = {
        'track': '01-eeg-to-image',
        'date': datetime.date.today().isoformat(),
        'mode': f'{choice} - {MODES[choice][0]}',
        'stage': 'smoke-B (real EEG, arbitrary labels, local harness)',
        'target_encoder': 'facebook/dinov2-small (competition uses dinov2-giant)',
        'metric': 'top-5 retrieval, full gallery',
        'eeg_source': source,
        'top5': round(report.top5, 3),
        'chance_top5': round(report.chance_top5, 3),
        'gallery_size': report.gallery_size,
        'inference_s': round(report.inference_s, 2),
        'external_pretraining': ['facebook/dinov2-small'],
        'compute_estimate_gpu_h': 0.0,
        'caveat': 'real EEG, arbitrary epoch-to-image labels; proves the contract, not decoding',
    }
    out = SAVE_DIR / f"smokeB_{record['date']}.json"
    out.write_text(json.dumps(record, indent=2))
    print(json.dumps(record, indent=2))
    print('\nwritten to', out)
else:
    print('skipped: choose mode 1 or higher')

---

# Stage 1 — NeuralBench, GPU, competition harness

**This is not a couple of data points.** NeuralBench's downloader works at corpus granularity, so
`--download` fetches all of Alljoined-1.6M. The Stage 0 budget does not apply. Expect 7.7 GB of EEG,
4.5 GB for DINOv2-giant, plus the prepare cache.

**Two stops, not one run.**

1. **Through the config-confirmation cell, then stop.** Storage is the live risk: env vars are not
   read, and an unconfigured run puts everything in `/tmp/neuralbench`, which Colab discards on
   restart. Confirm `DATA_DIR` resolves to Drive before starting a 7.7 GB transfer.
2. **Download, prepare, then `--debug -m eegnet`.** Prepare is the long part: it runs DINOv2 over
   every unique stimulus and is the only `--prepare` of the four tracks needing a GPU. The docs
   quote its timings across 10-128 SLURM jobs; on Colab it runs serially, so expect materially
   longer. `--debug` afterwards is the actual end-to-end proof and takes minutes.

EEGNet before REVE, because EEGNet needs no gated weights and gets you a green pipeline while the
HuggingFace approval for `brain-bzh/reve-base` is pending.

**Still inert:** the `check_model` call, the subset and gallery guards, and the inference timer. The
chance control is now real (`-m chance`). A score from this pass has few gates behind it, so treat
it as evidence the plumbing works and nothing more.

**Verified CLI surface** (from the help output above):

```
neuralbench [opts] {eeg,emg,fmri,meg} {task}
  --dataset D   loads datasets/{D}.yaml over the base config
  -m MODEL      chance dummy eegnet atcnet deep4net eegconformer shallow_fbcsp_net ctnet
                bendr biot cbramod labram luna reve mae  |  groups: all_classic all_fm all_baseline
  -w PRESET     linear_probe_flatten linear_probe_mean attentive_probe
                lora_r4_flatten lora_r32_flatten finetune_mean finetune_flatten
                (foundation models only)
  -d --debug    smaller config, runs locally, infra.mode='force'
  -p --prepare / --download / --plot-cached / -g --grid / -f --force / -r --retry
```

Three corrections to what this notebook assumed:

- **`chance` is a model.** The chance control needs no hand-wiring; `-m chance` puts it through the
  real scoring path. That turns the G1 anchor from a stub into a genuine check.
- **LoRA presets are `lora_r4_flatten` and `lora_r32_flatten`**, not `-w lora`. Two ranks to compare.
- **LUNA's CLI name is `luna`**, not `NtLuna`.

`image` is registered for `Gifford2022Large, Grootswagers2022Human, Xu2024Alljoined, Xu2025Alljoined`,
confirming `--dataset xu2025alljoined`.

Still unverified: nothing in the CLI sets DATA_DIR / CACHE_DIR / SAVE_DIR, so the env vars above are
still a guess. If the download lands somewhere unexpected, `config_manager.setup_config` is the
documented alternative.

⚠️ The install downgraded packages Colab pins (`requests`, `decorator`, `opentelemetry`). Harmless
in this session because Drive was already mounted, but after a runtime restart `google.colab`
imports may break. Mount Drive before installing, as this notebook does.

---
## [3] Data — Alljoined-1.6M

Registered as `xu2025alljoined`, confirmed present in the task's dataset list alongside
`Gifford2022Large`, `Grootswagers2022Human` and `Xu2024Alljoined`. 20 participants, 32 ch @ 256 Hz.

The size guard below exists because the *default* dataset for this task is THINGS-EEG2 at
**220 GB**. Omitting `--dataset` starts that download. Do not omit it.

In [ ]:
import shutil

DATASET = 'xu2025alljoined'   # NOT the default. Default = Gifford2022Large @ 220 GB.

for label, path, need in (('DATA_DIR (Drive)', DATA_DIR, 7.7 + 4.5),
                          ('CACHE_DIR (local)', CACHE_LOCAL, 6.0)):
    free = shutil.disk_usage(path).free / 1e9
    status = 'ok' if free > need else 'TOO SMALL'
    print(f'{label:20} {str(path):34} free {free:6.1f} GB  need ~{need:4.1f} GB  {status}')
    assert free > need, f'{label} has {free:.1f} GB free, needs ~{need:.1f} GB'

print('\nDownload is resumable: it skips files already on disk.')

In [ ]:
if RUN_HEAVY:
    !neuralbench eeg image --dataset {DATASET} --download
else:
    print('skipped: set RUN_HEAVY = True in Setup to fetch ~7.7 GB')

### [3b] Prepare the cache

**This is the only `--prepare` of the four tracks that needs a GPU** — it runs DINOv2-giant over
every unique stimulus to build the frozen target embeddings, alongside the usual window
preprocessing. Embeddings are content-keyed and shared across image tasks, so this cost is paid
once.

The docs quote SLURM-parallel timings (10 and 128 jobs). On Colab this runs serially and in-process,
so expect it to take proportionally longer.

In [ ]:
import time

if RUN_HEAVY:
    t0 = time.time()
    !neuralbench eeg image --dataset {DATASET} --prepare
    print(f'prepare took {(time.time() - t0) / 60:.1f} min')
else:
    print('skipped: needs RUN_HEAVY and a completed download')

---
## [6] Chance control

`chance` is a registered model, so this runs through the same scoring path as any other entry rather
than needing a hand-wired predictor. That makes it a genuine check on the retrieval and metric code.

Published chance on THINGS-EEG2 is Top-5 **2.22 ± 0.31**, consistent with its 200-image test gallery
(5/200 = 2.5%). Chance scales as `k / gallery_size`, so the Alljoined figure will differ. What
matters is whether the returned value matches `5 / gallery_size` for whatever gallery the split
produces. If it does not, the scoring path is wrong and nothing downstream is trustworthy.

In [ ]:
if RUN_HEAVY:
    !neuralbench eeg image --dataset {DATASET} -m chance
else:
    print('skipped: chance control needs the prepared cache')

---
## [7] Baseline — EEGNet

0.04 M params, ~2.5 h per seed on THINGS-EEG2 (expect less on the smaller Alljoined corpus).
This becomes **our** local reference on this corpus, since no published Alljoined number exists.

In [ ]:
if RUN_HEAVY and RUN_FULL_TRAIN:
    t0 = time.time()
    !neuralbench eeg image --dataset {DATASET} -m eegnet
    print(f'eegnet took {(time.time() - t0) / 60:.1f} min')
else:
    print('skipped: full training. Set RUN_FULL_TRAIN = True once the debug run is green.')

In [ ]:
if RUN_HEAVY and RUN_FULL_TRAIN:
    !neuralbench eeg image --dataset {DATASET} -m eegnet --plot-cached
else:
    print('skipped: nothing cached to plot yet')

---
## [8] The encoder: REVE

Selected on pretraining breadth, which is the property that should drive this choice.

| Encoder | Corpora | Scale | Channel handling |
|---|---|---|---|
| **REVE** | **92 datasets** | 60,000+ h, 25,000 subjects | 4D positional encoding over 3-D coords |
| LaBraM | 16 datasets | ~2,534 h | fixed 128-ch 10-20 order, unmatched channels dropped |
| BIOT | 6 datasets | - | Conv1d projection to 18 TCP bipolar channels |
| LUNA | TUEG + Siena | TUEG ~26,000 h | learned cross-attention unification |
| CBraMod | TUEG only | TUEG ~26,000 h | accepts any channel count |
| BENDR | TUEG only | TUEG ~26,000 h | fixed 20 channels |

REVE has 92 datasets against LaBraM's 16 and everyone else's 1-6. Three of the six are single-corpus
TUEG models, and TUEG is clinical pathology-screening EEG - far from healthy-subject viewing of
natural images, on top of being narrow.

**Montage handling is not a separate concern to defer.** You cannot pretrain across 92 heterogeneous
datasets without first solving montage invariance; they share no electrode layout. The narrow models
are TUEG-only precisely because one corpus with one montage is the easy case. REVE's positional
encoding is not a convenience bolted on - it is why the wide pretraining was possible.

The alternative has a concrete cost on this track: Alljoined is 32-channel consumer Emotiv. A
fixed-montage encoder either drops unmatched electrodes or interpolates onto a montage that was never
recorded. On 32 channels neither is affordable.

On leakage: REVE's image-task overlap is THINGS-EEG2, which taints its *published* 84.75. It never saw
Alljoined - what we train on, and where the hidden cohort comes from. The model is fine; the number is not.

**Second arm: LUNA.** Native 256 Hz matches Alljoined exactly where REVE resamples from 200 Hz, and it
reaches topology-invariance by a different mechanism. Useful as a check that results are not an artifact
of one encoder's inductive bias.

**Outside the zoo: ST-EEGFormer**, the KU Leuven model that won Challenge 1 in 2025 - roughly 13,300 h
across 11 datasets, open weights, proven in competition. Fewer corpora than REVE and it means leaving the
harness, so: back pocket, not starting point.

**Where the tuning effort probably belongs: the head and loss.** The target space is prescribed (frozen
DINOv2-giant, depth 0.6667, mean-pooled) and the stock config aligns to it with `ClipLoss`
(`norm_kind: y`, `temperature: false`, `symmetric: false`). Published EEG-to-image work (NICE, ATM,
NeuroCLIP) mostly varies the alignment head and objective rather than the backbone. `SigLipLoss` and
`DiffusionPrior` are already in the zoo.

Adaptation order: `-w linear_probe_mean` as a floor, then `-w lora_r4_flatten` and
`-w lora_r32_flatten`. The 2025 winner found full fine-tuning overfit while LoRA did not, and
EEG-FM-Compass found linear probing alone frequently insufficient, so run both ends. Two LoRA ranks
ship, which makes the rank sweep one extra flag rather than a code change. `-w` applies to
foundation models only.

In [ ]:
# login() blocks waiting for input, which stalls Run All. Run it on its own
# once REVE access is granted.
# from huggingface_hub import login; login()

In [ ]:
# REVE needs the gated brain-bzh/reve-base weights. Off Run All until access lands.
# !neuralbench eeg image --dataset {DATASET} -m reve -w linear_probe_mean
# !neuralbench eeg image --dataset {DATASET} -m reve -w lora_r4_flatten
# !neuralbench eeg image --dataset {DATASET} -m reve -w lora_r32_flatten
# !neuralbench eeg image --dataset {DATASET} -m luna -w lora_r4_flatten